**Avvio locale e Colab:** preparare la cartella completa seguendo il [README del laboratorio](../README.md). Conservare gli identificativi delle celle e salvare gli output nel proprio runtime.

<!-- C01 -->
# Computer Vision · Lezione 18
# Foundation Vision Studio
19 novembre 2026. Ogni etichetta Cxx conta tutte le celle, incluse le celle di testo. Eseguire nello stesso runtime. Le feature DINOv3 richiedono un checkpoint autorizzato personale. La corsa di riferimento SAM 2.1/LRASPP è CPU locale. Nessuna corsa Colab/GPU viene dichiarata.

Nei materiali del corso si usa il **punto decimale** (0.5) e la **virgola seguita da uno spazio** fra componenti di vettori e argomenti, per esempio (1.2, 1, −0.3). I conteggi interi non usano separatori delle migliaia.


**Lessico iniziale.** Feature o embedding: vettore di rappresentazione; patch: regione locale; backbone: estrattore. Prompt: indicazione del target, qui punto o box. Student: rete aggiornata; teacher: rete che fornisce i target; EMA: media mobile dei parametri. Norma L2: lunghezza, usata per normalizzare prima del coseno e della Gram. Support: esempi annotati per costruire i prototipi; query: casi separati da valutare. Few-shot usa un piccolo support; zero-shot richiede di dichiarare l’assenza di esempi per adattamento e l’eventuale assistenza dei prompt oracle.

<!-- C02 -->
## Percorso
C03–C07: ambiente e sei immagini. C09–C16: feature e DINOv3. C19–C31: prompt e failure cases. C33–C38: modello specializzato, metriche e costo. C40–C44: salvataggio e ripristino. Tre TODO, in C09, C23 e C35. Prima dell'esecuzione eseguire C03/C05 con rete disponibile.

In [ ]:
# C03
from pathlib import Path
import os,sys,subprocess,zipfile,importlib.util,json,platform,io
if importlib.util.find_spec('google') and importlib.util.find_spec('google.colab'):
    from google.colab import files
    colab_root=Path('/content/ComputerVision_Lezione18')
    def lesson_packages():
        return [p for p in colab_root.glob('**/supporto/foundation_utils.py')
                if (p.parent.parent/'manifest.json').is_file()
                and json.loads((p.parent.parent/'manifest.json').read_text()).get('lesson')==18]
    candidates=lesson_packages()
    if not candidates:
        print('Carica Pacchetto_studenti_Lezione_18.zip in questo runtime.')
        uploaded=files.upload()
        for name,payload in uploaded.items():
            if name.lower().endswith('.zip'):
                with zipfile.ZipFile(io.BytesIO(payload)) as z:
                    assert all(not n.startswith('/') and '..' not in Path(n).parts for n in z.namelist())
                    z.extractall(colab_root)
        candidates=lesson_packages()
    if len(candidates)!=1:raise RuntimeError('Serve un solo pacchetto valido della Lezione 18 nella cartella Colab dedicata.')
    ROOT=candidates[0].parent.parent
else:
    ROOT=Path.cwd()
    while not (ROOT/'supporto/foundation_utils.py').exists() and ROOT!=ROOT.parent: ROOT=ROOT.parent
    assert (ROOT/'supporto/foundation_utils.py').exists(),'Aprire dalla cartella notebook'
sys.path.insert(0,str(ROOT/'supporto'))
import torch,torchvision,numpy as np
assert tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2])>=(2,5),'Serve torch >=2.5'
# Conservare torch/torchvision del runtime. Installare solo dipendenze leggere mancanti.
for module,package in [('hydra','hydra-core==1.3.2'),('iopath','iopath==0.1.10')]:
    if importlib.util.find_spec(module) is None:subprocess.check_call([sys.executable,'-m','pip','install',package])
DEPS=Path(os.environ.get('CV18_DEPS',str(ROOT/'cache')));DEPS.mkdir(parents=True,exist_ok=True)
SAM_REPO=DEPS/'sam2'
if not (SAM_REPO/'sam2/build_sam.py').exists():
    subprocess.check_call(['git','clone','https://github.com/facebookresearch/sam2.git',str(SAM_REPO)])
    subprocess.check_call(['git','-C',str(SAM_REPO),'checkout','2b90b9f5ceec907a1c18123530e92e794ad901a4'])
actual=subprocess.check_output(['git','-C',str(SAM_REPO),'rev-parse','HEAD'],text=True).strip()
assert actual=='2b90b9f5ceec907a1c18123530e92e794ad901a4','Versione SAM diversa'
sys.path.insert(0,str(SAM_REPO))
from foundation_utils import *
DEVICE=torch.device(os.environ.get('CV_DEVICE','cuda' if torch.cuda.is_available() else 'cpu'))
assert DEVICE.type in ['cpu','cuda']
if DEVICE.type=='cuda' and not torch.cuda.is_available():raise RuntimeError('CUDA non disponibile: scegliere CPU o un runtime con GPU.')
torch.set_num_threads(1);torch.manual_seed(20261119)
OUT=ROOT/'output_lezione18';OUT.mkdir(exist_ok=True)
print({'device':str(DEVICE),'torch':torch.__version__,'torchvision':torchvision.__version__,'python':platform.python_version(),'threads':torch.get_num_threads(),'sam_commit':actual})

<!-- C04 -->
## Input, cache e provenienza
Il protocollo usa SAM 2.1 tiny pubblico Apache 2.0 e LRASPP MobileNetV3 con pesi torchvision. PennFudan è il dataset degli autori, da usare secondo il suo README. Lo ZIP didattico non ridistribuisce pesi o dataset originale. La cache Colab è temporanea. N01: perché aprire un altro notebook di setup non basta?

In [ ]:
# C05
import urllib.request
fetch(SAM_URL,DEPS/'sam2.1_hiera_tiny.pt','7402e0d864fa82708a20fbd15bc84245c2f26dff0eb43a4b5b93452deb34be69')
fetch(LR_URL,DEPS/'lraspp_mobilenet_v3_large-d234d4ea.pth','d234d4eae9d55d5f76de18b77cf0dc62c66fe5c5482758209d00f950c92bb280')
DATA=Path(os.environ.get('CV18_DATA',str(ROOT/'dati/PennFudanPed')))
if not (DATA/'PNGImages').exists():
    archive=fetch('https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip',ROOT/'dati/PennFudanPed.zip')
    with zipfile.ZipFile(archive) as z:
        assert all(not n.startswith('/') and '..' not in Path(n).parts for n in z.namelist())
        z.extractall(ROOT/'dati')
samples=[fixed_sample(DATA,name) for name in NAMES]
print('Immagini fissate:',NAMES)

<!-- C06 -->
## Sei immagini, due compiti
Per i prompt valutiamo l'istanza di area maggiore. Per il confronto semantico usiamo l'unione di tutte le persone annotate. I box e i punti del protocollo derivano dalle annotazioni: sono assistenza oracle controllata. Nessun addestramento sulle sei immagini. Non sono un benchmark rappresentativo né un test indipendente dal pretraining.

In [ ]:
# C07
import matplotlib.pyplot as plt
fig,axs=plt.subplots(2,3,figsize=(15,8))
for ax,s in zip(axs.flat,samples):
    ax.imshow(s['rgb']);ax.set_title(f"{s['name']} · {len(s['ids'])} istanze");ax.axis('off')
plt.tight_layout();plt.show()
print([(s['name'],s['target_id'],s['box'].tolist()) for s in samples])

<!-- C08 -->
## N02 · Similarità e normalizzazione
DINOv3 produce vettori, non nomi di classi. La similarità coseno confronta direzioni. C09: normalizzare le righe, rifiutare vettori nulli e calcolare la matrice tra due insiemi. C10 usa dati numerici didattici, non feature di una rete.

In [ ]:
# C09
def cosine_rows(a,b):
    a=np.asarray(a,dtype=float);b=np.asarray(b,dtype=float)
    # TODO C09: normalizzare le righe e calcolare a_normalizzato @ b_normalizzato.T.
    raise NotImplementedError('C09: similarità coseno')

In [ ]:
# C10
toy=np.array([[3.,4.],[0.,5.],[-3.,-4.]])
S=cosine_rows(toy,toy)
assert np.allclose(np.diag(S),1) and np.isclose(S[0,1],.8) and np.isclose(S[0,2],-1)
print('Esempio numerico didattico, non output DINOv3:\n',S)

<!-- C11 -->
## DINOv3: checkpoint autorizzato
Per eseguire il modello occorre ottenere l'accesso attraverso il sito ufficiale. Nessun token è richiesto o salvato qui. C12 accetta il percorso locale del proprio checkpoint ViT-S/16. Se manca, C13–C14 dichiarano «non eseguito» e si analizza la figura ufficiale nella presentazione. Non si usano pesi casuali e non si sostituisce il modello con DINOv2.

In [ ]:
# C12
DINO_CHECKPOINT=os.environ.get('DINO_CHECKPOINT','')  # File ViT-S/16 già ottenuto con autorizzazione.
DINO_REPO=os.environ.get('DINO_REPO','')              # Repository ufficiale locale.
dino_status='not_executed_no_authorized_checkpoint'
dino_features=None
if DINO_CHECKPOINT:
    assert Path(DINO_CHECKPOINT).is_file(),'Checkpoint inesistente'
    assert DINO_REPO and (Path(DINO_REPO)/'hubconf.py').is_file(),'Indicare repository ufficiale DINOv3'
    dino_status='configured_not_yet_executed'
print(dino_status)

In [ ]:
# C13
if DINO_CHECKPOINT:
    from torchvision.transforms import v2
    transform=v2.Compose([v2.ToImage(),v2.Resize((256,256),antialias=True),v2.ToDtype(torch.float32,scale=True),v2.Normalize((.485,.456,.406),(.229,.224,.225))])
    dino=torch.hub.load(DINO_REPO,'dinov3_vits16',source='local',weights=DINO_CHECKPOINT).eval().to(DEVICE)
    with torch.inference_mode():
        result=dino.forward_features(transform(samples[0]['image']).unsqueeze(0).to(DEVICE))
    dino_features=result['x_norm_patchtokens'][0].float().cpu().numpy()
    assert dino_features.shape==(256,384),dino_features.shape
    dino_status='executed_authorized_checkpoint'
    print('Patch 16×16, dimensione 384:',dino_features.shape)
else:
    print('DINOv3 non eseguito: analisi delle figure degli autori e del calcolo coseno.')

In [ ]:
# C14
if dino_features is not None:
    # Punto dell'istanza maggiore convertito nella griglia 16×16.
    s=samples[0];x,y=s['point'];h,w=s['gt'].shape;col=min(15,int(x/w*16));row=min(15,int(y/h*16));idx=row*16+col
    similarity=cosine_rows(dino_features[idx:idx+1],dino_features)[0].reshape(16,16)
    plt.figure(figsize=(7,5));plt.imshow(similarity,vmin=-1,vmax=1,cmap='viridis');plt.colorbar(label='Coseno');plt.scatter(col,row,c='red',marker='+');plt.title('DINOv3, checkpoint autorizzato');plt.show()
    np.savez_compressed(OUT/'dinov3_features.npz',features=dino_features,similarity=similarity)
else:print('Nessuna mappa DINOv3 locale disponibile.')

<!-- C15 -->
## N03 · Prototipi e transfer
C16 usa un piccolo esempio esplicito per verificare nearest prototype. I prototipi vanno stimati su support distinto dalle query. Se sono disponibili feature DINOv3, progettare uno split per immagine prima di qualsiasi adattamento. PCA e colori delle mappe non costituiscono classi semantiche.

In [ ]:
# C16
support=np.array([[1.,0.],[.8,.2],[0.,1.],[.2,.8]])
labels=np.array([0,0,1,1]);queries=np.array([[.9,.1],[.1,.9],[.5,.5]])
normalized=support/np.linalg.norm(support,axis=1,keepdims=True)
prototypes=np.stack([normalized[labels==c].mean(0)for c in [0,1]])
scores=cosine_rows(queries,prototypes);prediction=scores.argmax(1)
print('Esempio numerico didattico:',scores,prediction)
print('Terza query: parità teorica, argmax sceglie il primo indice; non è certezza.')

<!-- C17 -->
## Discussione DINOv3
Il teacher EMA fornisce target di auto-supervisione. Le loss immagine e patch non assegnano nomi umani. Gram anchoring conserva le relazioni tra patch tramite le matrici dei prodotti scalari delle feature normalizzate. Le celle DINOv3 si attivano solo con un checkpoint autorizzato. Stato del report esplicito, senza metriche fittizie.

<!-- C18 -->
## SAM 2.1 su immagini statiche
Image encoder una volta per immagine, prompt encoder a ogni interazione, mask decoder per le maschere candidate. Qui non si propaga una memoria temporale. Usiamo FP32, niente compilazione e nessun riempimento di buchi con estensione CUDA. CPU e CUDA sono entrambi consentiti.

In [ ]:
# C19
predictor=load_sam(DEPS,DEVICE)
print('SAM 2.1 Hiera tiny, parametri:',sum(p.numel()for p in predictor.model.parameters()))

In [ ]:
# C20
s=samples[0]
_,encode_seconds=timed(lambda:predictor.set_image(s['rgb']),DEVICE)
print('Dimensione originale',s['rgb'].shape,'embedding principale',tuple(predictor.get_image_embedding().shape),'encoder secondi',encode_seconds)

<!-- C21 -->
## N04 · Un punto e un box
Le coordinate sono (x, y) in pixel originali. Il predictor applica il proprio resize a 1024×1024 e la normalizzazione. Un punto positivo ha label 1, uno negativo label 0. Il box usa x1, y1, x2, y2. Il modello può restituire tre maschere per un punto ambiguo.

In [ ]:
# C22
with torch.inference_mode():
    candidates,scores,lowres=predictor.predict(point_coords=s['point'][None],point_labels=np.array([1]),multimask_output=True)
    point_mask=candidates[int(scores.argmax())].astype(bool)
    box_masks,box_scores,_=predictor.predict(box=s['box'],multimask_output=False)
print('Score predetti dei tre candidati:',scores,'selezionato:',int(scores.argmax()))
show_masks(s,[point_mask,box_masks[0].astype(bool)],['Punto positivo','Box stretto'],points_by_panel=[[(s['point'],1)],[]],boxes_by_panel=[[],[s['box']]]);plt.show()

In [ ]:
# C23
def mask_stats(pred,gt):
    pred=np.asarray(pred,dtype=bool);gt=np.asarray(gt,dtype=bool)
    assert pred.shape==gt.shape
    # TODO C23: intersezione, unione, IoU e Dice. Convenzione: entrambe vuote -> 1.
    raise NotImplementedError('C23: metriche maschera')

In [ ]:
# C24
assert np.isclose(mask_stats(np.array([1,1,0]),np.array([1,0,1]))['iou'],1/3)
assert np.isclose(mask_stats(np.array([1,1,0]),np.array([1,0,1]))['dice'],.5)
print('Punto:',mask_stats(point_mask,s['gt']))
print('Box:',mask_stats(box_masks[0],s['gt']))

<!-- C25 -->
## N05 · Score predetto e metrica
Lo score SAM stima la qualità. L'IoU misurata confronta con la maschera annotata. La selezione usa soltanto argmax dello score predetto. C26 calcola l'IoU di tutti i candidati per una diagnosi successiva, senza cambiare il candidato selezionato.

In [ ]:
# C26
candidate_iou=[mask_stats(m,s['gt'])['iou'] for m in candidates]
print({'predicted_scores':scores.tolist(),'measured_iou':candidate_iou,'selected_by_score':int(scores.argmax())})
show_masks(s,[m.astype(bool)for m in candidates],[f'Candidato {j}, score {scores[j]:.3f}, IoU {candidate_iou[j]:.3f}' for j in range(3)]);plt.show()

<!-- C27 -->
## N06 · Sensibilità al prompt
Prima delle misure fissiamo: box stretto, box allargato del 25% della larghezza/altezza su ciascun lato e limitato all'immagine, stesso box allargato più punto positivo e negativo. Il negativo deriva dai falsi positivi della prima maschera: è una correzione oracle controllata, non una procedura automatica senza GT. La politica non garantisce miglioramento.

In [ ]:
# C28
records=[];predictions={}
for sample in samples:
    row,arrays=run_sam_sample(predictor,sample,DEVICE)
    for mode in ['point','box','loose_box','corrected']:
        row['prompts'][mode].update(mask_stats(arrays[mode],sample['gt']))
    row['point_candidate_ious']=[mask_stats(m,sample['gt'])['iou']for m in arrays['point_candidates']]
    records.append(row);predictions[sample['name']]=arrays
    print(sample['name'],{k:round(v['iou'],4)for k,v in row['prompts'].items()})

In [ ]:
# C29
modes=['point','box','loose_box','corrected']
values=np.array([[r['prompts'][m]['iou']for m in modes]for r in records])
print('Media semplice delle 6 IoU, non IoU dei pixel aggregati:',dict(zip(modes,values.mean(0))))
fig,ax=plt.subplots(figsize=(11,5))
for j,mode in enumerate(modes):ax.plot(range(6),values[:,j],marker='o',label=mode)
ax.set_xticks(range(6),[n.replace('Ped000',' ')for n in NAMES]);ax.set_ylim(0,1);ax.set_ylabel('IoU istanza');ax.legend();ax.grid(alpha=.2);plt.show()

<!-- C30 -->
## N07 · Correzione e failure cases
Il primo caso mostra un recupero, il secondo un peggioramento con la politica fissata. Non selezionare a posteriori solo i miglioramenti. Una correzione può modificare quale oggetto o quale parte SAM interpreta come target. L'annotazione può avere bordi e convenzioni differenti.

In [ ]:
# C31
for sample,row in zip(samples[:2],records[:2]):
    a=predictions[sample['name']]
    show_masks(sample,[a['loose_box'],a['corrected']],[f"Box allargato, IoU {row['prompts']['loose_box']['iou']:.3f}",f"Correzione, IoU {row['prompts']['corrected']['iou']:.3f}"],points_by_panel=[[],[(sample['point'],1),(row['negative_point'],0)]],boxes=[sample['loose_box']])
    plt.show()

<!-- C32 -->
## Salvataggio e controllo dello stato
Le variabili restano nel runtime. Se il runtime termina, usare la procedura di recupero e distinguere gli output di riferimento da una nuova esecuzione.

In [ ]:
# C33
specialized,preprocess=load_lraspp(DEPS,DEVICE)
for sample,row in zip(samples,records):
    mask,seconds=run_lraspp(specialized,preprocess,sample,DEVICE)
    predictions[sample['name']]['lraspp']=mask;row['lraspp_seconds']=seconds
print('LRASPP, classe VOC person=15, inferenza completata.')

<!-- C34 -->
## N08 · Confronto con LRASPP
LRASPP produce una classe per pixel senza prompt. SAM usa un box GT per ogni persona e la nostra unione rende comparabile l'output semantico. Il costo informativo è diverso: riportare il numero di box e non chiamare questo esperimento una classifica paritaria di modelli. Il dataset può non annotare tutte le persone visibili.

In [ ]:
# C35
def union_masks(masks):
    masks=np.asarray(masks,dtype=bool)
    assert masks.ndim==3
    # TODO C35: OR logico sull'asse delle istanze.
    raise NotImplementedError('C35: unione semantica')

In [ ]:
# C36
assert np.array_equal(union_masks(np.array([[[1,0]],[[0,1]]])),np.array([[True,True]]))
for sample,row in zip(samples,records):
    a=predictions[sample['name']];a['sam_union']=union_masks(a['sam_instances'])
    row['semantic']={k:mask_stats(a[k],sample['labels']>0)for k in ['lraspp','sam_union']}
    print(sample['name'],row['instances'],{k:round(v['iou'],4)for k,v in row['semantic'].items()})

In [ ]:
# C37
print('Media per immagine:',{k:float(np.mean([r['semantic'][k]['iou']for r in records]))for k in ['lraspp','sam_union']})
for sample in [samples[2],samples[3]]:
    a=predictions[sample['name']]
    # Per il confronto semantico il contorno giallo è l'unione delle annotazioni.
    sem=dict(sample);sem['gt']=sample['labels']>0
    show_masks(sem,[a['lraspp'],a['sam_union']],['LRASPP senza prompt','SAM con box GT per ogni istanza']);plt.show()

In [ ]:
# C38
cost={'device':str(DEVICE),'sam_encoder_seconds':[r['sam_encoder_seconds']for r in records],'sam_box_decoders_seconds':[r['sam_union_decoder_seconds']for r in records],'lraspp_total_seconds':[r['lraspp_seconds']for r in records]}
print(cost)
print('SAM: encoder più N decoder. LRASPP: preprocess più forward e resize output. Lettura immagini e download esclusi. Singolo passaggio per immagine, non benchmark con warm-up.')

<!-- C39 -->
## Limiti del confronto
Sei scene selezionate per illustrare metodi e fallimenti, non campionamento rappresentativo. Nessun training locale. SAM riceve coordinate oracle, LRASPP una tassonomia fissa. Il pretraining può contenere dati correlati. CPU locale e CUDA hanno costi e possibili differenze numeriche. DINOv3 ha stato separato.

In [ ]:
# C40
missing=[name for name in ['records','predictions','cost','dino_status'] if name not in globals()]
if missing:raise RuntimeError('Prima del salvataggio completare C12, C28, C33, C35/C36 e C38. Mancano: '+', '.join(missing))
if [row['name'] for row in records]!=NAMES or any('semantic' not in row or not all(k in row['semantic'] for k in ['lraspp','sam_union']) for row in records):
    raise RuntimeError('C40 richiede le sei misure semantiche complete di C36, dopo C28 e C33.')
if any(name not in predictions or not all(k in predictions[name] for k in ['labels','sam_instances','sam_union','lraspp']) for name in NAMES):
    raise RuntimeError('Maschere incomplete: eseguire C28, C33 e C36 prima di esportare.')
report={'course':'Computer Vision','lesson':18,'date':'2026-11-19','seed':20261119,'device':str(DEVICE),'threads':torch.get_num_threads(),'torch':torch.__version__,'torchvision':torchvision.__version__,'python':platform.python_version(),'platform':platform.platform(),'sam_commit':actual,'sam_sha256':sha(DEPS/'sam2.1_hiera_tiny.pt'),'lraspp_sha256':sha(DEPS/'lraspp_mobilenet_v3_large-d234d4ea.pth'),'dinov3_status':dino_status,'protocol':'six fixed PennFudan images; largest instance for prompts; GT-assisted coordinates; LRASPP semantic person versus SAM union from all GT boxes; no local training','records':records,'cost':cost}
if DINO_CHECKPOINT:report['dinov3_sha256']=sha(DINO_CHECKPOINT)
(OUT/'metriche_finali.json').write_text(json.dumps(report,indent=2))
for name,arrays in predictions.items():np.savez_compressed(OUT/f'{name}.npz',**arrays)
print('Output salvati:',OUT)

<!-- C41 -->
## N09 · Foundation card
Due pagine: F01 dati/pesi e autorizzazioni. F02 prompt e costo informativo. F03 metriche per immagine e aggregazione. F04 un recupero e un peggioramento spiegati. F05 stato DINOv3, limiti e prossimo confronto controllato. Allegare JSON e notebook eseguito. I risultati di riferimento restano etichettati come tali.

In [ ]:
# C42
if not (OUT/'metriche_finali.json').is_file():
    raise RuntimeError('Nessun report salvato: eseguire C40 dopo avere completato il protocollo.')
export_record=json.loads((OUT/'metriche_finali.json').read_text())
if export_record.get('lesson')!=18 or [r['name'] for r in export_record.get('records',[])]!=NAMES or any('semantic' not in r for r in export_record['records']):
    raise RuntimeError('Report incompleto: completare C36/C38 e ripetere C40.')
export_names=['metriche_finali.json']+[f'{name}.npz' for name in NAMES]
if export_record.get('dinov3_status')=='executed_authorized_checkpoint':export_names.append('dinov3_features.npz')
missing=[name for name in export_names if not (OUT/name).is_file()]
if missing:raise RuntimeError('Output mancanti, ripetere C14 se DINOv3 è attivo e C40: '+', '.join(missing))
archive=ROOT/'Output_Lezione_18.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for name in export_names:z.write(OUT/name,name)
print(archive)
if importlib.util.find_spec('google') and importlib.util.find_spec('google.colab'):
    from google.colab import files
    files.download(str(archive))
print('Salvare anche il notebook dal menu File: non è incluso automaticamente nello ZIP output.')

<!-- C43 -->
## Discussione conclusiva
E01: coseno alto equivale a stessa classe? E02: score SAM 0.9 significa IoU misurata 0.9? E03: quale conclusione permette il confronto oracle SAM/LRASPP? Rispondere prima della correzione collettiva.

In [ ]:
# C44
reloaded=json.loads((OUT/'metriche_finali.json').read_text())
assert reloaded['sam_sha256']==sha(DEPS/'sam2.1_hiera_tiny.pt')
for row in reloaded['records']:
    with np.load(OUT/f"{row['name']}.npz")as a:
        measured=mask_stats(a['sam_union'],a['labels']>0)['iou']
    assert abs(measured-row['semantic']['sam_union']['iou'])<1e-12
print('Ripristino delle metriche da maschere salvate: verificato.')

<!-- C45 -->
## Chiusura
Il recupero degli output non ricalcola il modello. Per una nuova inferenza ripartire da C03/C05/C19 e rieseguire l'esperimento. Per DINOv3 conservare il proprio checkpoint autorizzato, il percorso e l'hash senza pubblicare token o link personali.